In [54]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

def highlight_max(row):
    is_max = row == row.max()
    return ['font-weight: bold' if v else '' for v in is_max]

def drop_nan_group(group):
    if group.isnull().any():
        return None  # Returning None will drop the group
    else:
        return group

def compte_les_points(df):
    results = {}
    total = 0
    for idx, row in df.reset_index()['perf'].iterrows():
        is_max = row == row.max()
        for key in is_max.reset_index()['package'].values:
            if key not in results:
                results[key] = 0
                
            results[key] += int(is_max[key])
        total += 1
            
    for key in results:
        print(f"{key} : {results[key]} / {total}")

In [55]:
# Load bench 
bench = pd.read_csv("/Users/rudy/Documents/CHU/iias/baiddy_group/automl/src/perf_logger/tests_data/bench_v2.log")

In [56]:
bench['perf'] = bench.apply(lambda row: row['balanced_accuracy'] if pd.notnull(row['balanced_accuracy']) else row['r2_score'], axis=1)

# With folds

In [66]:
merged_df = bench[['package', 'max_duration', 'dataset', 'perf', 'fold']]
grouped = merged_df.groupby(['package', 'max_duration', 'dataset', 'fold']).mean(numeric_only=True).reset_index()[['package', 'perf', 'max_duration', 'dataset', 'fold']].dropna()

# grouped
pivot_df = grouped.pivot(index=['dataset', 'max_duration', 'fold'], columns='package').dropna()
pivot_df.style.apply(highlight_max, axis=1)

# Mean of folds

In [58]:
merged_df = bench[['package', 'max_duration', 'dataset', 'perf', 'compute_time']]
grouped = merged_df.groupby(['package', 'max_duration', 'dataset']).mean(numeric_only=True).reset_index()[['package', 'perf', 'max_duration', 'dataset']].dropna()

# grouped
pivot_df = grouped.pivot(index=['dataset', 'max_duration'], columns='package').dropna()
pivot_df.style.apply(highlight_max, axis=1)

In [59]:
# def count(row):
#     is_max = row == row.max()
#     return ['font-weight: bold' if v else '' for v in is_max]

# pivot_df = grouped.pivot(index=['dataset_name', 'duration'], columns='package')

# pivot_df.style.apply(highlight_max, axis=1)

compte_les_points(pivot_df)

FEDOT : 0 / 1
automed : 1 / 1
flaml : 0 / 1
naive_autoML : 0 / 1


In [60]:
# Auto-Sklearn : 14 / 75
# AutoMed : 29 / 75
# NaiveAutoML : 38 / 75

In [61]:
# PAR DATASET sans prendre ne compte le temps
without_duration = grouped.drop(columns=['max_duration'])
without_duration

pivot_df = without_duration.groupby(['package', 'dataset']).max(numeric_only=True).reset_index().pivot(index=['dataset'], columns='package').dropna()
pivot_df.style.apply(highlight_max, axis=1)

# Attention Naive_AutoML à probablement plus de temps ci-dessus

In [62]:
compte_les_points(pivot_df)

FEDOT : 0 / 1
automed : 1 / 1
flaml : 0 / 1
naive_autoML : 0 / 1


In [63]:

import matplotlib.pyplot as plt
import seaborn as sns

def line_plot(df, filter_tuple, hue="dataset_name",zero_line=False, y_lim=None):
    plt.figure()
    plot = sns.lineplot(data=df[df[filter_tuple[0]] == filter_tuple[1]], x="duration", y="perf", hue=hue) 
    sns.move_legend(plot, "upper left", bbox_to_anchor=(1, 1))
    plt.title(str(filter_tuple))
    if zero_line:
        plt.axhline(y=0.0, color='r', linestyle='--')
        
    if y_lim:
        plt.ylim(*y_lim)
    plt.show()


merged_df = pd.concat([naive, automed_void, autosk])[['package', 'duration', 'dataset_name', 'perf']]
merged_df = merged_df.dropna() # Avoid penalize a package if no value


NameError: name 'naive' is not defined

In [ ]:
line_plot(merged_df, ("package", "AutoMed_void"))

In [ ]:
line_plot(merged_df, ("package", "NaiveAutoML"))

In [ ]:
line_plot(merged_df, ("package", "Auto-Sklearn"))

In [ ]:
for dataset_name in merged_df['dataset_name'].unique():
    line_plot(merged_df, ("dataset_name", dataset_name), hue="package")

In [ ]:
# Diff with automed ?
# Ensure df is a copy to avoid SettingWithCopyWarning
df = merged_df.copy()

# Separate 'automed' and other packages
automed_df = df[df['package'] == 'AutoMed_void']
other_df = df[df['package'] != 'AutoMed_void']

# Average 'automed' performance for each 'duration' and 'dataset_name' combination
automed_avg_perf = automed_df.groupby(['duration', 'dataset_name'])['perf'].max()

# Merge this average performance back into the other_df based on 'duration' and 'dataset_name'
other_df = other_df.merge(automed_avg_perf, on=['duration', 'dataset_name'], suffixes=('', '_automed_avg'))

# Calculate the difference in performance
other_df['perf'] = other_df['perf'] - other_df['perf_automed_avg']

# Drop the now unnecessary 'perf_automed_avg' column
other_df.drop(columns=['perf_automed_avg'], inplace=True)

# Resulting DataFrame has 'automed' rows removed and 'perf' adjusted
df_result = other_df

In [ ]:
for dataset_name in merged_df['dataset_name'].unique():
    try:
        line_plot(df_result, ("dataset_name", dataset_name), hue="package", zero_line=True, y_lim=(-0.2, 0.2))
    except:
        print(f"error for {dataset_name}")